## Importing the appropriate Libraries for the Project


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
TARGET_USD = 30000
EXCHANGE_RATE = 1500

In [ ]:
historical_data = pd.read_csv('statement.csv')

In [ ]:
historical_data.head()

In [ ]:
historical_data.info()

In [ ]:
print(historical_data.isnull().sum())

print(historical_data.isnull().sum().sum())

## Data Cleaning

In [ ]:
# List the columns we want to clean
cols_to_clean = ['Debit(NGN)', 'Credit(NGN)', 'Balance After(NGN)']

for col in cols_to_clean:
    # 1. Remove commas
    historical_data[col] = historical_data[col].str.replace(',', '', regex=False)
    
    # 2. Replace the dashes with '0'
    historical_data[col] = historical_data[col].str.replace('--', '0', regex=False)
    
    # 3. Convert the cleaned string into a proper float (decimal number)
    historical_data[col] = historical_data[col].astype(float)

# Fill any true NaNs in Debit/Credit with 0 just in case
historical_data['Debit(NGN)'] = historical_data['Debit(NGN)'].fillna(0)
historical_data['Credit(NGN)'] = historical_data['Credit(NGN)'].fillna(0)

# Check the results!
historical_data.head()



In [ ]:
##Check the information of the column
historical_data.info()

In [ ]:
historical_data['Balance After(USD)'] = historical_data['Balance After(NGN)'] / EXCHANGE_RATE

print(historical_data['Balance After(USD)'].head())

## Data Processing

In [ ]:
historical_data.info()

In [ ]:
print(historical_data['Trans. Date'].head())

In [ ]:
# 1. Extract the time into a brand new column
historical_data['Time'] = pd.to_datetime(historical_data['Trans. Date']).dt.time

# 2. Convert 'Value Date' into a proper datetime object (it only has the date, no time)
historical_data['Value Date'] = pd.to_datetime(historical_data['Value Date'])

# 3. Set 'Value Date' as your powerful datetime index
historical_data.set_index('Value Date', inplace=True)

# 4. Delete the original 'Trans. Date' column since we don't need it anymore
historical_data.drop(columns=['Trans. Date'], inplace=True)

historical_data.head()


In [ ]:
historical_data.info()

## Monthly Resampling & Core Metrics

In [ ]:
monthly_data = historical_data.resample('ME').sum(numeric_only=True)

In [ ]:
monthly_data.tail()

In [ ]:
monthly_data['monthly_net_savings'] = monthly_data['Credit(NGN)'] - monthly_data['Debit(NGN)']
avg_monthly_expenses = monthly_data['Debit(NGN)'].mean()
avg_monthly_savings = monthly_data['monthly_net_savings'].mean()

## Calculate Runway & Time-to-Goal


In [ ]:
historical_data.head()

In [ ]:
# Pull the very last snapshot for both currencies
current_balance_usd = historical_data['Balance After(USD)'].iloc[-1]
current_balance_ngn = historical_data['Balance After(NGN)'].iloc[-1]

print(f"Current Balance (NGN): ₦{current_balance_ngn:,.2f}")
print(f"Current Balance (USD): ${current_balance_usd:,.2f}")


In [ ]:
standard_runway = current_balance_ngn / avg_monthly_expenses
print(standard_runway)

In [ ]:
avg_monthly_savings_usd = avg_monthly_savings / EXCHANGE_RATE

projected_time_to_goal = (TARGET_USD - current_balance_usd) / avg_monthly_savings_usd
print(projected_time_to_goal)


In [ ]:
print("="*40)
print(" 💰 SAVINGS ENGINE REPORT 💰")
print("="*40)
print(f"Standard Runway:       {standard_runway:.2f} Months")
print(f"Projected Time to Goal: {projected_time_to_goal:,.2f} Months")
print("-" * 40)
print(f"Avg Monthly Savings:   ${avg_monthly_savings_usd:.2f}")
print("="*40)

## Visualizations

In [ ]:
# 1. Set the size of the canvas (width, height)
plt.figure(figsize=(12, 6))

# 2. Draw the line chart using seaborn
# Since your dates are the index, we use x=historical_data.index
sns.lineplot(data=historical_data, x=historical_data.index, y='Balance After(USD)', color='blue', linewidth=1.5, estimator=None)


# 3. Add titles and labels to make it look official
plt.title('Wealth Snapshot: Account Balance Over Time (USD)', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Balance (USD)', fontsize=12)

# 4. Rotate the date labels slightly so they don't overlap with each other
plt.xticks(rotation=45)

# 5. Clean up the layout and display the chart!
plt.tight_layout()
plt.show()

In [ ]:
monthly_data.head()

In [ ]:
# 1. Create a copy of the data and format the index to strings BEFORE plotting
plot_data = monthly_data[['Credit(NGN)', 'Debit(NGN)']].copy()
plot_data.index = plot_data.index.strftime('%Y-%m')

# 2. Plot directly from the new dataframe
plot_data.plot(kind='bar', color=['green', 'red'], figsize=(14, 7))

# 3. Add formatting
plt.title('Monthly Cash Flow: Income vs. Expenses', fontsize=16, fontweight='bold')
plt.xlabel('Month', fontsize=12)
plt.ylabel('Amount (NGN)', fontsize=12)
plt.legend(['Income', 'Expenses'])

# 4. Rotate labels and show
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
